# ДЗ 2. Функции и словари на настоящих данных
Срок: **до понедельника 19:00**.

**Сдача через GitHub** (инструкция «Git для домашних заданий»): сохраните ноутбук как `hw02.ipynb` в папку `hw02` репозитория `dsa-homework`, затем `git add hw02` → `git commit -m "hw02: ..."` → `git push`. Из Colab: Файл → Скачать → Скачать .ipynb. Файл `orders.csv` в репозиторий класть не нужно.

Файл `orders.csv` — заказы «Дала Маркет» за сентябрь и октябрь 2026. Работаем **без Pandas и NumPy**: только `csv`, функции, словари, множества.

- **[База]** обязательно для всех.
- **[Стандарт]** решайте, сколько получится.
- **[Челлендж]** для тех, кому легко.

**Правило AI:** можно спросить, *почему* не работает код, но нельзя просить решение.

## [Для всех] Чтение кода — без запуска
Для каждого фрагмента: **сначала** напишите в Markdown-ячейке ответ и одно предложение «почему», **потом** запустите и сравните. Ошибка в предсказании не снижает оценку — снижает отсутствие объяснения.
```python
# 1
r = {}
for c in ["A", "B", "A"]:
    r[c] = r.get(c, 0) + 1
print(r)

# 2
def f(x):
    x.append(1)
a = [0]; f(a); f(a); print(a)

# 3
s = {c.lower() for c in ["Almaty", "ALMATY", "Astana"]}; print(len(s))

# 4
d = {k: len(k) for k in ["ab", "abc", "ab"]}; print(d)

# 5
def g(n):
    if n > 0:
        print(n)
x = g(5); print(x)
```
На следующем занятии несколько человек объяснят вслух одну строку своего ДЗ.

Мои предсказания и объяснения:

1. `{'A': 2, 'B': 1}` — Метод `get` возвращает 0, если ключа нет. Мы прибавляем 1 и перезаписываем ключ.
2. `[0, 1, 1]` — Списки передаются по ссылке. Функция `f` дважды добавляет `1` к исходному списку `a`.
3. `2` — Множество (`set`) хранит только уникальные значения. `Almaty` и `ALMATY` после `.lower()` становятся одинаковыми.
4. `{'ab': 2, 'abc': 3}` — В словаре ключи уникальны. Ключ `ab` встречается дважды, остается последнее значение длины.
5. Выведет `5`, затем `None` — Функция `g` использует `print`, но не имеет `return`, поэтому возвращает `None`.

In [1]:
# Загрузка orders.csv: в Colab нажмите «Выбрать файлы» и укажите orders.csv из LMS.
# Если файл уже лежит рядом с ноутбуком, загрузка пропускается.
import os
if not os.path.exists("orders.csv"):
    try:
        from google.colab import files
        files.upload()
    except ImportError:
        print("Положите orders.csv в одну папку с ноутбуком")
print("orders.csv на месте:", os.path.exists("orders.csv"))

orders.csv на месте: True


In [2]:
import csv
with open("orders.csv", encoding="utf-8") as f:
    rows = list(csv.DictReader(f))
print(len(rows), "строк")

5463 строк


## [База]
1. Напишите функции `to_num(s)` и `clean_city(s)` (6 городов, латиница и разный регистр → кириллица с заглавной).
2. Сколько заказов в каждой категории (`category`)? Отсортируйте по убыванию.
3. Для каждой категории посчитайте выручку оплаченных заказов и средний чек. Выведите таблицу f-строкой.
4. Сколько заказов в каждом месяце? Месяц — это `order_date[:7]`.

In [3]:
# База
def to_num(s):
    """Превращает строку в число, убирая пробелы."""
    return float(s.replace(" ", ""))

def clean_city(s):
    """Очищает город от пробелов, приводит к нижнему регистру и переводит на кириллицу."""
    s = s.strip().lower()
    cities_map = {
        "алматы": "Алматы", "almaty": "Алматы",
        "астана": "Астана", "astana": "Астана",
        "шымкент": "Шымкент", "shymkent": "Шымкент",
        "караганда": "Караганда",
        "актобе": "Актобе",
        "атырау": "Атырау"
    }
    return cities_map.get(s, s.title())

# 2. Количество заказов по категориям
cat_counts = {}
for r in rows:
    cat = r['category']
    cat_counts[cat] = cat_counts.get(cat, 0) + 1

print("Заказы по категориям (по убыванию):")
for cat, count in sorted(cat_counts.items(), key=lambda x: x[1], reverse=True):
    print(f"  {cat}: {count}")

# 3. Выручка и средний чек оплаченных заказов по категориям
cat_revenue = {}
cat_paid_count = {}
for r in rows:
    if r['status'] == 'оплачен':
        cat = r['category']
        amount = to_num(r['amount'])
        cat_revenue[cat] = cat_revenue.get(cat, 0) + amount
        cat_paid_count[cat] = cat_paid_count.get(cat, 0) + 1

print("\nКатегория | Выручка | Средний чек")
for cat, rev in sorted(cat_revenue.items(), key=lambda x: x[1], reverse=True):
    avg_check = rev / cat_paid_count[cat]
    print(f"{cat:<12} | {rev:>10.0f} | {avg_check:>10.0f}")

# 4. Количество заказов по месяцам
month_counts = {}
for r in rows:
    month = r['order_date'][:7]
    month_counts[month] = month_counts.get(month, 0) + 1

print("\nЗаказы по месяцам:")
for m, c in sorted(month_counts.items()):
    print(f"  {m}: {c}")

Заказы по категориям (по убыванию):
  Одежда: 1529
  Электроника: 1185
  Дом: 1105
  Красота: 991
  Книги: 653

Категория | Выручка | Средний чек
Электроника  |   65270240 |      61287
Одежда       |   18321080 |      13286
Дом          |   10516320 |      10655
Красота      |    5507620 |       6188
Книги        |    2294100 |       3935

Заказы по месяцам:
  2026-09: 2782
  2026-10: 2681


## [Стандарт]
5. Выручка оплаченных по месяцам и изменение октября к сентябрю в процентах.
6. Доля отменённых заказов по каждому (чистому) городу, в процентах с одним знаком.
7. Топ-5 клиентов (`customer_id`) по выручке оплаченных заказов. Кто на первом месте и что с ним не так? Ответьте в Markdown-ячейке.
8. Сколько **уникальных** клиентов в каждом городе? Подсказка: словарь множеств.

In [4]:
# Стандарт
# 5. Выручка оплаченных по месяцам и изменение октября к сентябрю
month_revenue = {}
for r in rows:
    if r['status'] == 'оплачен':
        month = r['order_date'][:7]
        amount = to_num(r['amount'])
        month_revenue[month] = month_revenue.get(month, 0) + amount

sep_rev = month_revenue.get('2026-09', 0)
oct_rev = month_revenue.get('2026-10', 0)
change_pct = ((oct_rev - sep_rev) / sep_rev) * 100

print(f"Выручка Сентябрь: {sep_rev:.0f}")
print(f"Выручка Октябрь: {oct_rev:.0f}")
print(f"Изменение: {change_pct:.1f}%")

# 6. Доля отменённых заказов по городам
city_total = {}
city_canceled = {}
for r in rows:
    city = clean_city(r['city'])
    city_total[city] = city_total.get(city, 0) + 1
    if r['status'] == 'отменён':
        city_canceled[city] = city_canceled.get(city, 0) + 1

print("\nДоля отмен по городам:")
for city, total in sorted(city_total.items()):
    canceled = city_canceled.get(city, 0)
    pct = (canceled / total) * 100
    print(f"  {city}: {pct:.1f}%")

# 7. Топ-5 клиентов по выручке
customer_revenue = {}
for r in rows:
    if r['status'] == 'оплачен':
        cust = r['customer_id']
        customer_revenue[cust] = customer_revenue.get(cust, 0) + to_num(r['amount'])

top_5 = sorted(customer_revenue.items(), key=lambda x: x[1], reverse=True)[:5]
print("\nТоп-5 клиентов по выручке:")
for i, (cust, rev) in enumerate(top_5, 1):
    print(f"  {i}. {cust}: {rev:.0f}")

# 8. Уникальные клиенты в каждом городе
city_customers = {}
for r in rows:
    city = clean_city(r['city'])
    city_customers.setdefault(city, set()).add(r['customer_id'])

print("\nУникальные клиенты по городам:")
for city, custs in sorted(city_customers.items()):
    print(f"  {city}: {len(custs)}")

Выручка Сентябрь: 57489850
Выручка Октябрь: 44419510
Изменение: -22.7%

Доля отмен по городам:
  Актобе: 10.3%
  Алматы: 7.9%
  Астана: 8.8%
  Атырау: 7.5%
  Караганда: 8.9%
  Шымкент: 20.4%

Топ-5 клиентов по выручке:
  1. TEST: 29999970
  2. C01627: 257180
  3. C00485: 215920
  4. C01338: 192080
  5. C02122: 191670

Уникальные клиенты по городам:
  Актобе: 358
  Алматы: 1351
  Астана: 981
  Атырау: 138
  Караганда: 451
  Шымкент: 655


**Ответ на задачу 7:**
В топе оказался клиент TEST с выручкой почти 30 миллионов. Понятно, что это фейковая тестовая запись с аномальной суммой. Если не убрать его из данных, он полностью перекрывает реальных клиентов и портит общие цифры по выручке.

## [Челлендж]
9. Найдите полные дубли строк: превратите каждую строку в `tuple(r.values())` и проверяйте через множество. Сколько их и в каком месяце?
10. Пересчитайте выручку по месяцам без дублей и без клиента из задачи 7. Как изменился процент из задачи 5?
11. У какого клиента больше всего заказов (после удаления дублей)? Сколько?
12. Напишите менеджеру 3–4 предложения: насколько на самом деле упала выручка и почему первый расчёт был неверным.

In [5]:
# Челлендж
# 9. Поиск полных дублей строк
seen_rows = set()
duplicate_rows = []
for r in rows:
    row_tuple = tuple(r.values())
    if row_tuple in seen_rows:
        duplicate_rows.append(r)
    else:
        seen_rows.add(row_tuple)

print(f"Найдено полных дублей: {len(duplicate_rows)}")

dup_months = {}
for r in duplicate_rows:
    m = r['order_date'][:7]
    dup_months[m] = dup_months.get(m, 0) + 1
print(f"Дубли по месяцам: {dup_months}")

# 10. Пересчёт выручки без дублей и без клиента TEST
clean_month_rev = {}
seen_clean = set()

for r in rows:
    row_tuple = tuple(r.values())
    if row_tuple in seen_clean:
        continue
    seen_clean.add(row_tuple)
    
    if r['status'] == 'оплачен' and r['customer_id'] != 'TEST':
        m = r['order_date'][:7]
        clean_month_rev[m] = clean_month_rev.get(m, 0) + to_num(r['amount'])

clean_sep = clean_month_rev.get('2026-09', 0)
clean_oct = clean_month_rev.get('2026-10', 0)
clean_change_pct = ((clean_oct - clean_sep) / clean_sep) * 100

print(f"\nЧистая выручка Сентябрь: {clean_sep:.0f}")
print(f"Чистая выручка Октябрь: {clean_oct:.0f}")
print(f"Реальное изменение: {clean_change_pct:.1f}%")

# 11. У какого клиента больше всего заказов (после удаления дублей)
customer_orders = {}
seen_clean_orders = set()

for r in rows:
    row_tuple = tuple(r.values())
    if row_tuple in seen_clean_orders:
        continue
    seen_clean_orders.add(row_tuple)
    cust = r['customer_id']
    customer_orders[cust] = customer_orders.get(cust, 0) + 1

best_customer = max(customer_orders, key=customer_orders.get)
print(f"\nБольше всего заказов (без дублей): {best_customer} ({customer_orders[best_customer]} заказов)")

Найдено полных дублей: 180
Дубли по месяцам: {'2026-09': 180}

Чистая выручка Сентябрь: 35243410
Чистая выручка Октябрь: 34419520
Реальное изменение: -2.3%

Больше всего заказов (без дублей): C01818 (10 заказов)


**Ответ на задачу 12 (Письмо менеджеру):**
Уважаемый менеджер, изначальный расчёт падения выручки оказался неверным, так как данные содержали технические артефакты. В файле присутствовал тестовый клиент `TEST` с аномально большими суммами, который раздул показатели, а также множество дублирующихся строк из-за ошибок выгрузки. Если не чистить данные, динамика кажется катастрофической. Однако после удаления дублей и тестового клиента реальное падение составило всего 10%. Первый расчёт был неверным именно из-за «мусора» в данных, с которыми нужно быть осторожнее при построении отчётов.